# KuroSiwo to VIIRS Showcase

This notebook demonstrates a minimal CLI-driven workflow for KuroSiwo-driven VIIRS extraction. It does three things:

1. Loads the KuroSiwo GeoPackage catalogue from `assets/ks_catalogue.gpkg`
2. Calls the Atlantis CLI to derive the metadata CSV and fetch VIIRS data for one case
3. Loads the resulting metadata and rasters for inspection and visualisation

Before running it, make sure you have:

- installed the notebook dependencies with `uv sync --extra notebooks`
- pulled the Git LFS assets with `git lfs pull`
- network access to the public NOAA JPSS S3 bucket at `https://noaa-jpss.s3.amazonaws.com`
- optional: access to `https://jpssflood.gmu.edu/downloads/pub` only if you want to force the legacy GMU backend

In [ ]:
from pathlib import Path
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Run this notebook from inside the Atlantis repository.')


repo_root = find_repo_root(Path.cwd().resolve())
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

drafts_dir = repo_root / 'notebooks' / 'drafts'
scratch_dir = drafts_dir / 'data' / 'viirs_showcase'
scratch_dir.mkdir(parents=True, exist_ok=True)

catalogue_path = repo_root / 'assets' / 'ks_catalogue.gpkg'
selected_case = 'KuroSiwo_1111005'
days_before = 0
days_after = 0
viirs_backend = 'noaa_s3'
viirs_format = 'tif'
refresh_metadata_cache = False
metadata_cache_tag = f'{viirs_backend}_{viirs_format}_before_{days_before}_after_{days_after}'
metadata_output_path = scratch_dir / 'kurosiwo_metadata_cli.csv'
fetch_root = scratch_dir / selected_case / 'viirs'

print(f'repo_root: {repo_root}')
print(f'catalogue_path: {catalogue_path}')
print(f'scratch_dir: {scratch_dir}')
print(f'selected_case: {selected_case}')
print(f'viirs_backend: {viirs_backend}')
print(f'viirs_format: {viirs_format}')
print(f'metadata_output_path: {metadata_output_path}')
print(f'fetch_root: {fetch_root}')
print(f'refresh_metadata_cache: {refresh_metadata_cache}')

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import rioxarray as rxr
from shapely.geometry import box

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.max_columns = 20
pd.options.display.width = 140

## 1. Build the Atlantis metadata CSV via the CLI

This notebook keeps the Python layer thin and lets the Atlantis CLI produce the metadata CSV.

Set `refresh_metadata_cache = True` if you want to regenerate the CSV instead of reusing the cached file in `notebooks/drafts/data/viirs_showcase/`.

In [ ]:
catalogue = gpd.read_file(catalogue_path)
if refresh_metadata_cache or not metadata_output_path.exists():
    command = [
        sys.executable,
        '-m',
        'atlantis.cli',
        'build-kurosiwo-metadata',
        '--catalogue',
        str(catalogue_path),
        '--output',
        str(metadata_output_path),
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=repo_root, check=True)
    cache_status = 'rebuilt via CLI'
else:
    cache_status = 'loaded from cache'

metadata = pd.read_csv(metadata_output_path, parse_dates=['date_start', 'date_end'])
print(f'catalogue rows: {len(catalogue):,}')
print(f'distinct events: {catalogue["actid"].nunique()}')
print(f'metadata cache: {metadata_output_path.relative_to(repo_root)} ({cache_status})')
metadata.head()

In [ ]:
top_events = metadata.nlargest(10, 'max_flood_extent_km2').sort_values('max_flood_extent_km2')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_events['flood_case'], top_events['max_flood_extent_km2'], color='#2f6c8f')
ax.set_title('Largest KuroSiwo events by derived flood extent')
ax.set_xlabel('max_flood_extent_km2')
ax.set_ylabel('')
plt.show()

## 2. Inspect one KuroSiwo case

Choose one flood case from the derived metadata CSV, inspect its bounds, then fetch the matching VIIRS data through the CLI.

In [ ]:
selected_event = metadata.loc[metadata['flood_case'] == selected_case].iloc[0]
selected_actid = int(selected_case.split('_')[-1])
event_rows = catalogue.to_crs('EPSG:4326')
event_rows = event_rows[event_rows['actid'] == selected_actid].copy()
event_bbox = box(
    selected_event['lon_min'],
    selected_event['lat_min'],
    selected_event['lon_max'],
    selected_event['lat_max'],
)

selected_event[
    ['flood_case', 'date_start', 'date_end', 'max_flood_extent_km2']
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
event_rows.boundary.plot(ax=ax, linewidth=0.3, color='#6baed6', alpha=0.7)
gpd.GeoSeries([event_bbox], crs='EPSG:4326').boundary.plot(ax=ax, color='#d95f02', linewidth=2)
ax.set_title(f'{selected_case}: catalogue footprints and derived event bbox')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

## 3. Fetch VIIRS through the CLI

This notebook delegates the fetch step to `fetch-kurosiwo-viirs`. By default Atlantis searches the NOAA JPSS public S3 bucket. Set `viirs_backend = 'gmu_legacy'` above if you want to compare against the older GMU source.

In [ ]:
command = [
    sys.executable,
    '-m',
    'atlantis.cli',
    'fetch-kurosiwo-viirs',
    '--catalogue',
    str(catalogue_path),
    '--case',
    selected_case,
    '--output',
    str(scratch_dir),
    '--days-before',
    str(days_before),
    '--days-after',
    str(days_after),
    '--viirs-backend',
    viirs_backend,
    '--viirs-format',
    viirs_format,
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=repo_root, check=True)
fetch_dir = fetch_root / 'processed'
if not fetch_dir.exists():
    raise RuntimeError(f'Expected processed output at {fetch_dir}')
sorted(path.name for path in fetch_dir.glob('*.tif'))

## 4. Visualise the fetched VIIRS layers

The CLI fetch writes three rasters per date into the processed directory: `flood_extent`, `quality_mask`, and `permanent_water`.

In [ ]:
processed_files = sorted(fetch_dir.glob('*.tif'))
if not processed_files:
    raise RuntimeError('No processed GeoTIFFs were written by the CLI fetch command.')

files_by_kind = {}
for path in processed_files:
    if path.name.endswith('_flood_extent.tif'):
        files_by_kind['flood_extent'] = path
    elif path.name.endswith('_quality_mask.tif'):
        files_by_kind['quality_mask'] = path
    elif path.name.endswith('_permanent_water.tif'):
        files_by_kind['permanent_water'] = path

dataset = pd.Series({
    name: rxr.open_rasterio(path).squeeze(drop=True)
    for name, path in files_by_kind.items()
}).to_dict()
dataset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
dataset['flood_extent'].plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Flood extent')
dataset['quality_mask'].plot(ax=axes[1], cmap='gray')
axes[1].set_title('Quality mask')
dataset['permanent_water'].plot(ax=axes[2], cmap='viridis')
axes[2].set_title('Permanent water')
for ax in axes:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
plt.show()